# [skip-ci]

# Cell-Type Enrichment with ViralScan

This vignette demonstrates two ways to add cell-type context to ViralScan results:

1. Use `--cell-types` during a normal `viralscan` run so the pipeline writes `results/cell_type_enrichment.tsv` automatically.
2. Use the importable `viralscan.enrichment` API in Python when you already have an AnnData object and a cell annotation CSV.

The notebook is self-contained: it can create a synthetic AnnData object if you do not have a completed ViralScan run available.

## Step 1 - Load AnnData (real or synthetic)

If you ran `basic_usage.ipynb` first, set `USE_REAL = True`. The default path below follows ViralScan's per-sample output layout: `viralscan_output/SRR20710651/kb-python/counts_unfiltered/adata_multimap.h5ad`.

Otherwise, the notebook creates a small synthetic AnnData object so the enrichment API can be demonstrated without a full workflow run.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp

USE_REAL = False
h5ad_path = (
    Path("viralscan_output")
    / "SRR20710651"
    / "kb-python"
    / "counts_unfiltered"
    / "adata_multimap.h5ad"
)

if USE_REAL and h5ad_path.exists():
    import scanpy as sc

    adata = sc.read_h5ad(h5ad_path)
    print(f"Loaded real AnnData: {adata}")
else:
    rng = np.random.default_rng(42)
    n_cells, n_genes = 200, 10
    barcodes = [f"CELL{i:04d}-1" for i in range(n_cells)]
    gene_names = [
        "GAPDH",
        "ACTB",
        "CD3D",
        "CD14",
        "MS4A1",
        "FCGR3A",
        "NKG7",
        "LYZ",
        "INFLUENZA_A_PB2",
        "INFLUENZA_A_NS1",
    ]
    counts = rng.negative_binomial(5, 0.5, size=(n_cells, n_genes)).astype(np.float32)
    viral_mask = rng.random((n_cells, 2)) > 0.9
    counts[:, 8:] = counts[:, 8:] * viral_mask
    adata = ad.AnnData(
        X=sp.csr_matrix(counts),
        obs=pd.DataFrame(index=barcodes),
        var=pd.DataFrame(index=gene_names),
    )
    print(f"Synthetic AnnData: {adata}")

## Step 2 - Prepare a barcode-to-cell-type CSV

The `--cell-types` CLI flag and the Python API both expect a CSV with `barcode` and `cell_type` columns. Barcodes must match `adata.obs_names` exactly, including suffixes such as `-1`.

In a real analysis these labels usually come from Seurat, scanpy, Azimuth, CellTypist, or a manual marker-gene annotation workflow. Here we assign mock labels.

In [ ]:
import tempfile

cell_type_labels = ["T cell", "B cell", "Monocyte", "NK cell"]
barcodes_list = adata.obs_names.tolist()
assigned = [cell_type_labels[i % len(cell_type_labels)] for i in range(len(barcodes_list))]

cell_types_df = pd.DataFrame({"barcode": barcodes_list, "cell_type": assigned})

tmp_dir = Path(tempfile.mkdtemp())
cell_types_csv = tmp_dir / "cell_types.csv"
cell_types_df.to_csv(cell_types_csv, index=False)
print(f"Cell-types CSV written to: {cell_types_csv}")
cell_types_df.head(8)

## Step 3 - Run `cell_type_enrichment()`

`group_by_virus` maps each display name to one or more viral feature names in `adata.var_names`. For a real ViralScan object, inspect `adata.var_names` and use the viral features present in your reference.

In [ ]:
from viralscan.enrichment import cell_type_enrichment, write_cell_type_enrichment

group_by_virus = {
    "Influenza A": ["INFLUENZA_A_PB2", "INFLUENZA_A_NS1"],
}

cfg = {"cell_types": str(cell_types_csv)}

enrichment_df = cell_type_enrichment(adata, group_by_virus, cfg)
print(f"Enrichment result shape: {enrichment_df.shape}")
enrichment_df

## Step 4 - Write the enrichment TSV

`write_cell_type_enrichment()` expects a sample run directory and writes into its `results/` subdirectory, matching the full ViralScan workflow.

In [ ]:
output_dir = tmp_dir / "viralscan_enrichment_demo"
output_dir.mkdir(exist_ok=True)

out_path = write_cell_type_enrichment(enrichment_df, str(output_dir))

if out_path:
    written = Path(out_path)
    print(f"TSV written: {written}")
    print(f"File size: {written.stat().st_size} bytes")
    reloaded = pd.read_csv(written, sep="\t")
    print(f"Reloaded rows: {len(reloaded)}")
else:
    print("Nothing written. The enrichment result was empty.")

## Step 5 - Bar chart of enrichment p-values

In [ ]:
import matplotlib.pyplot as plt

if enrichment_df.empty:
    print("No enrichment data to plot.")
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    values = -np.log10(enrichment_df["padj"] + 1e-10)
    colors = ["tab:red" if p < 0.05 else "tab:gray" for p in enrichment_df["padj"]]
    ax.bar(enrichment_df["cell_type"], values, color=colors)
    ax.axhline(-np.log10(0.05), color="red", linestyle="--", linewidth=0.8, label="padj = 0.05")
    ax.set_xlabel("Cell type")
    ax.set_ylabel("-log10(padj)")
    ax.set_title("Cell-type enrichment - Influenza A")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Summary and interpretation

The enrichment table contains one row per virus and cell-type combination:

| Column | Meaning |
|--------|---------|
| `n_infected` | Infected cells in this cell type |
| `n_total` | All labeled cells of this type |
| `pct` | Percent of cells in this type with at least one viral UMI |
| `OR` | Fisher exact odds ratio, one-sided for enrichment |
| `pvalue` | Raw Fisher p-value |
| `padj` | Benjamini-Hochberg adjusted p-value |

Cell types with `padj < 0.05` are over-represented among infected cells relative to all other labeled cells.

In a real dataset:

1. Inspect `adata.var_names` and build `group_by_virus` from viral features that actually appear in your ViralScan output.
2. Supply a real `barcode,cell_type` CSV from your cell annotation workflow.
3. Pass `--cell-types your_annotation.csv` to `viralscan` when you want the TSV and HTML report section generated automatically.